# Notebook Dedicated to create the list of candidates (GTID + runID + subrunID)

In [11]:
import numpy as np
import glob
import pickle
import pandas as pd
import seaborn as sn

# Functions

## Flat Lists

In [12]:
def flatten_list(xs):
    flat_list = []
    
    for element in xs:
        # Si el elemento es otra lista, llamamos a la función de nuevo (recursividad)
        if isinstance(element, list):
            flat_list.extend(flatten_list(element))
        # Si no es una lista (es decir, es un número), lo agregamos al resultado
        else:
            flat_list.append(element)
            
    return flat_list

# Load Data

## Load all Data Separately from Analysis15, Analysis15_bMR and Analysis20_bMR. Also Apply Cuts

#main_dir = '/home/joankl/data/solars/real_data/bisMSB/first_candidates/'
main_dir = 'E:/Data/solars/solarnu_Realdata/bisMSB/first_candidates/'
candidates_data_dir_list = glob.glob(main_dir + 'analysis*/ntuple/resume_files/')

runID_analysis15 = np.load(main_dir + 'analysis15/ntuple/resume_files/runID.npy')
subrunID_analysis15 = np.load(main_dir + 'analysis15/ntuple/resume_files/subrunID.npy')
eventID_analysis15 = np.load(main_dir + 'analysis15/ntuple/resume_files/eventID.npy')
energy_analysis15 = np.load(main_dir + 'analysis15/ntuple/resume_files/energy_corrected.npy')
posr_analysis15 = np.load(main_dir + 'analysis15/ntuple/resume_files/posr_av.npy')

runID_analysis15_bMR = np.load(main_dir + 'analysis15_bMR/ntuple/resume_files/runID.npy')
subrunID_analysis15_bMR = np.load(main_dir + 'analysis15_bMR/ntuple/resume_files/subrunID.npy')
eventID_analysis15_bMR = np.load(main_dir + 'analysis15_bMR/ntuple/resume_files/eventID.npy')
energy_analysis15_bMR = np.load(main_dir + 'analysis15_bMR/ntuple/resume_files/energy_corrected.npy')
posr_analysis15_bMR = np.load(main_dir + 'analysis15_bMR/ntuple/resume_files/posr_av.npy')

runID_analysis20_bMR = np.load(main_dir + 'analysis20_bMR/ntuple/resume_files/runID.npy')
subrunID_analysis20_bMR = np.load(main_dir + 'analysis20_bMR/ntuple/resume_files/subrunID.npy')
eventID_analysis20_bMR = np.load(main_dir + 'analysis20_bMR/ntuple/resume_files/eventID.npy')
energy_analysis20_bMR = np.load(main_dir + 'analysis20_bMR/ntuple/resume_files/energy_corrected.npy')
posr_analysis20_bMR = np.load(main_dir + 'analysis20_bMR/ntuple/resume_files/posr_av.npy')


# ===== Apply General Cuts =====
en_cut = 5.0
posr_cut = 5500.0

condition_analysis15 = (energy_analysis15 >= 5) & (posr_analysis15 <= posr_cut)
condition_analysis15_bMR = (energy_analysis15_bMR >= 5) & (posr_analysis15_bMR <= posr_cut)
condition_analysis20_bMR = (energy_analysis20_bMR >= 5) & (posr_analysis20_bMR <= posr_cut)

runID_analysis15 = runID_analysis15[condition_analysis15]
eventID_analysis15 = eventID_analysis15[condition_analysis15]

runID_analysis15_bMR = runID_analysis15_bMR[condition_analysis15_bMR]
eventID_analysis15_bMR = eventID_analysis15_bMR[condition_analysis15_bMR]

runID_analysis20_bMR = runID_analysis20_bMR[condition_analysis20_bMR]
eventID_analysis20_bMR = eventID_analysis20_bMR[condition_analysis20_bMR]

#print(f'for Analysis15, the run ID range is: [{min(runID_analysis15)}, {max(runID_analysis15)}]')
#print(f'for Analysis15_bMR, the run ID range is: [{min(runID_analysis15_bMR)}, {max(runID_analysis15_bMR)}]')
#print(f'for Analysis20_bMR, the run ID range is: [{min(runID_analysis20_bMR)}, {max(runID_analysis20_bMR)}]')

# ===== Now Apply Hotspot and atm Cuts =====
# ------- HS and atm IDs -------
hs_eventID = []
hs_runID = []

atm_eventID = []
atm_runID = []

# Load HS and atm results
for candidates_data_dir_i in candidates_data_dir_list:
    with open(candidates_data_dir_i + 'hs_dict.pkl', 'rb') as f:
        hs_dict = pickle.load(f)
        hs_eventID.append(hs_dict['eventID'])
        hs_runID.append(hs_dict['runID'])
        
    with open(candidates_data_dir_i + 'atm_dict.pkl', 'rb') as f:
        atm_dict = pickle.load(f)
        atm_eventID.append(atm_dict['eventID'])
        atm_runID.append(atm_dict['runID'])

hs_eventID = flatten_list(hs_eventID)
hs_runID = flatten_list(hs_runID)
atm_eventID = flatten_list(atm_eventID)
atm_runID = flatten_list(atm_runID)

# Full runID and eventID to remove
reject_eventID = np.concatenate((hs_eventID, atm_eventID))
reject_runID = np.concatenate((hs_runID, atm_runID))

# Remove coincident eventID and runID events with tagged HS and atm eventID and runID
# Create an unique number such that runID*offset + eventID is an unique number

offset = np.int64(10**10)

unique_id_data = (runID.astype(np.int64) * offset) + eventID.astype(np.int64)
unique_id_hs_atm = (reject_runID.astype(np.int64) * offset) + reject_eventID.astype(np.int64)

In [ ]:
# Observabels to load
obs_list = ['energy_corrected', 'eventID', 'runID', 'subrunID']

# Define Directories
pattern_dir = '/home/joankl/data/solars/real_data/bisMSB/first_candidates/analysis*/resume_files/'
full_fdir = glob.glob(pattern_dir)

# Create empty dictionary to save the data
obs_dict = {var: np.array([]) for var in obs_list}

# Loop over the pattern dir:
for fdir_i in full_fdir:
    # Loop on the obs_list
    for obs_i in obs_list:
        obs_arr = np.load(fdir_i + obs_i + '.npy')

        #Save the observables
        obs_dict[obs_i] = np.append(obs_dict[obs_i], obs_arr)

# Extract data of interest and apply cuts
energy = obs_dict['energy_corrected']

en_cut = 5.0

condition = (energy >= en_cut)

energy = obs_dict['energy_corrected'][condition]
eventID = obs_dict['eventID'][condition]
runID = obs_dict['runID'][condition]
subrunID = obs_dict['subrunID'][condition]

## Load all Data at the same time and  Apply Cuts and remove atm and HS

In [21]:
# ------- Directory of data -------
candidates_data_dir = 'E:/Data/solars/solarnu_Realdata/bisMSB/first_candidates/analysis*/ntuple/resume_files/'
candidates_data_dir_list = glob.glob(candidates_data_dir)

# ------- Observable list -------
obs_list = ['energy_corrected', 'posr_av', 'runID', 'eventID']

# ------- Observable Dictionary -------
obs_dict = {obs: np.array([]) for obs in obs_list}

# ------- HS and atm IDs -------
hs_eventID = []
hs_runID = []

atm_eventID = []
atm_runID = []

for candidates_data_dir_i in candidates_data_dir_list:
    for obs in obs_list:
        obs_dir = candidates_data_dir_i + obs + '.npy'
        obs_i = np.load(obs_dir)
        obs_dict[obs] = np.append(obs_dict[obs], obs_i)

print(f'Nº of intial events : {len(obs_dict['energy_corrected'])}')

# ===== Apply General Cuts =====
en_inf_cut = 5
posr_cut = 5500

energy_condition = (obs_dict['energy_corrected'] >= en_inf_cut)
posr_condition = (obs_dict['posr_av'] <= posr_cut)

mask = (energy_condition & posr_condition) 

energy = obs_dict['energy_corrected'][mask]
posr_av = obs_dict['posr_av'][mask]
runID = obs_dict['runID'][mask]
eventID = obs_dict['eventID'][mask]

print(f'Nº of events after basic cuts: {len(energy)}')

# ===== Now Apply Hotspot and atm Cuts =====

# Load HS and atm results
for candidates_data_dir_i in candidates_data_dir_list:
    with open(candidates_data_dir_i + 'hs_dict.pkl', 'rb') as f:
        hs_dict = pickle.load(f)
        hs_eventID.append(hs_dict['eventID'])
        hs_runID.append(hs_dict['runID'])
        
    with open(candidates_data_dir_i + 'atm_dict.pkl', 'rb') as f:
        atm_dict = pickle.load(f)
        atm_eventID.append(atm_dict['eventID'])
        atm_runID.append(atm_dict['runID'])

hs_eventID = flatten_list(hs_eventID)
hs_runID = flatten_list(hs_runID)
atm_eventID = flatten_list(atm_eventID)

atm_runID = flatten_list(atm_runID)

# Full runID and eventID to remove
reject_eventID = np.concatenate((hs_eventID, atm_eventID))
reject_runID = np.concatenate((hs_runID, atm_runID))

# Remove coincident eventID and runID events with tagged HS and atm eventID and runID
# Create an unique number such that runID*offset + eventID is an unique number

offset = np.int64(10**10)

unique_id_data = (runID.astype(np.int64) * offset) + eventID.astype(np.int64)
unique_id_hs_atm = (reject_runID.astype(np.int64) * offset) + reject_eventID.astype(np.int64)

is_hs_atm = np.isin(unique_id_data, unique_id_hs_atm)

hs_atm_cut = ~is_hs_atm

# Select the data
eventID = eventID[hs_atm_cut]
runID = runID[hs_atm_cut]
print(f'Nº of events after atm and HS cuts: {len(eventID)}')

Nº of intial events : 26504
Nº of events after basic cuts: 192
Nº of events after atm and HS cuts: 186


## Load all Data Simultaneously

# Save Files

In [26]:
fdir_save = 'candidates list/'

## Create the Excel Folder

### All files together (Analysis15, Analysis15_bMR, Analysis20_bMR)

In [14]:
f_outname = f'filtered_solar_analysis_E_cut_{en_cut}_MeV_R_cut_5500_mm'  # Output file name

df = pd.DataFrame({'eventID': np.array(eventID, dtype = np.int64),
                   'runID': runID,
                   'subrunID': subrunID})

df.to_excel(f_outname + '.xlsx', index = False)

## Create txt list

### All files together (Analysis15, Analysis15_bMR, Analysis20_bMR)

In [27]:
data = np.column_stack((runID, eventID))
np.savetxt(fdir_save + f"filtered_solar_analysis_E_cut_{en_cut}_MeV_R_cut_5500_mm.txt", data,
           fmt="%d, %d",          # enteros
           delimiter=", ",
           comments="")

### Separated Files (Analysis15, Analysis15_bMR, Analysis20_bMR)

In [24]:
# Analysis15
fname = 'analysis15_candidatesID'
data = np.column_stack((runID_analysis15, eventID_analysis15))
np.savetxt(fdir_save + fname + ".txt", data,
           fmt="%d, %d",          # enteros
           delimiter=", ",
           comments="")

# Analysis15_bMR
fname = 'analysis15bMR_candidatesID'
data = np.column_stack((runID_analysis15_bMR, eventID_analysis15_bMR))
np.savetxt(fdir_save + fname + ".txt", data,
           fmt="%d, %d",          # enteros
           delimiter=", ",
           comments="")

# Analysis20_bMR
fname = 'analysis20bMR_candidatesID'
data = np.column_stack((runID_analysis20_bMR, eventID_analysis20_bMR))
np.savetxt(fdir_save + fname + ".txt", data,
           fmt="%d, %d",          # enteros
           delimiter=", ",
           comments="")